## Importing the Data

In [1]:
import pandas as pd
import numpy as np
import os

collegeScorecard = pd.read_csv(
    "../../../data/processed/CS/College_Scorecard_cleaned.csv",
    dtype={"CIP_family": str}
)
pseo = pd.read_csv(
    "../../../data/processed/PSEO/PSEO_cleaned.csv",
    dtype={"CIP_family": str, "CIP_series": str}
)

pseo_cip = set(pseo["CIP_family"].dropna())
pseo_series = set(pseo["CIP_series"].dropna())

print("CollegeScorecard shape:", collegeScorecard.shape)
print("PSEO shape:", pseo.shape)

CollegeScorecard shape: (26678, 107)
PSEO shape: (18826, 69)


## Basic Analysis before Merge

Finding the unique CIP_family values from each dataset:

In [2]:
cs_cip = set(collegeScorecard["CIP_family"].dropna())
pseo_cip = set(pseo["CIP_family"].dropna())

print("CollegeScorecard unique CIP_family values:", len(cs_cip))
print("PSEO unique CIP_family values:", len(pseo_cip))
print("Overlapping CIP_family values:", len(cs_cip & pseo_cip))

CollegeScorecard unique CIP_family values: 360
PSEO unique CIP_family values: 319
Overlapping CIP_family values: 307


Checking which values are in one dataset but not the other:

In [3]:
only_in_cs = sorted(cs_cip - pseo_cip)
only_in_pseo = sorted(pseo_cip - cs_cip)

print("\nIn CollegeScorecard but not PSEO:", len(only_in_cs))
print(only_in_cs)

print("\nIn PSEO but not CollegeScorecard:", len(only_in_pseo))
print(only_in_pseo)


In CollegeScorecard but not PSEO: 53
['01.82', '03.99', '04.02', '04.08', '04.10', '04.99', '05.99', '12.06', '13.03', '13.07', '14.06', '14.11', '14.22', '14.23', '14.33', '14.37', '14.41', '14.48', '15.14', '16.14', '16.17', '19.04', '19.99', '25.99', '26.04', '26.07', '26.08', '27.99', '28.06', '29.99', '30.08', '30.22', '30.25', '30.29', '30.36', '30.41', '30.44', '30.46', '30.51', '30.52', '30.53', '30.70', '39.99', '40.02', '45.13', '48.00', '50.11', '51.01', '51.04', '51.12', '51.16', '53.02', '61.26']

In PSEO but not CollegeScorecard: 12
['14.03', '14.32', '14.39', '14.40', '14.45', '26.05', '29.03', '33.01', '35.01', '37.01', '45.05', '60.01']


In [4]:
collegeScorecard["has_pseo_match"] = collegeScorecard["CIP_family"].isin(pseo_cip)

match_pct = collegeScorecard["has_pseo_match"].mean() * 100
print(f"\n% of CollegeScorecard rows with a CIP_family found in PSEO: {match_pct:.1f}%")


% of CollegeScorecard rows with a CIP_family found in PSEO: 99.5%


C:\Users\andre\AppData\Local\Temp\ipykernel_39000\2047420818.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  collegeScorecard["has_pseo_match"] = collegeScorecard["CIP_family"].isin(pseo_cip)


In [5]:
missing_rows = collegeScorecard[~collegeScorecard["has_pseo_match"]]
print("\nTop CIP_family values among unmatched rows:")
print(missing_rows["CIP_family"].value_counts().head(10))


Top CIP_family values among unmatched rows:
CIP_family
30.70    18
16.17    10
13.03     7
51.16     6
39.99     5
48.00     5
27.99     4
30.22     4
30.25     3
26.07     3
Name: count, dtype: int64


## Things to Note:

- Format matches exactly — both use the same `XX.YY` style (e.g. `16.09`), so no reformatting was needed.
- Of our 360 unique CIP_family codes, 307 have a match in PSEO.
- 53 CIP_family codes in our data don't appear in PSEO at all — any program under one of these codes will end up with no earnings data after merging, not because of missing/suppressed values, but because PSEO simply doesn't cover that field of study.
- At the row level, **99.5% of our actual rows** have a CIP_family that exists in PSEO — this is the number that matters more than the code-level count above, since it reflects how much of our real data would survive the merge.

-----
## Possible fix:

- Comparing our CIP codes against PSEO's, 99.51% of our rows get an exact match on the full CIP code. Only 0.47% don't match exactly but could still get a close estimate by matching on the shortened (broader category) CIP code instead.
- These program-level matches are only used for the **national benchmark columns**. They never replace the school + program earnings (see below).


In [6]:
# Row-level match breakdown: family-level match, series-level fallback, or no match at all
collegeScorecard["CIP_series"] = collegeScorecard["CIP_family"].str[:2]

def match_level(cip_family):
    if pd.isna(cip_family):
        return "none"
    if cip_family in pseo_cip:
        return "family"
    elif cip_family[:2] in pseo_series:
        return "series"
    else:
        return "none"

collegeScorecard["cip_match_level"] = collegeScorecard["CIP_family"].apply(match_level)

match_counts = collegeScorecard["cip_match_level"].value_counts()
match_pcts = collegeScorecard["cip_match_level"].value_counts(normalize=True) * 100

print(pd.DataFrame({"rows": match_counts, "pct": match_pcts.round(2)}))

                  rows    pct
cip_match_level              
family           26546  99.51
series             126   0.47
none                 6   0.02


**NOTE:** Looking at this analysis, just 0.02% (6 rows) don't match at all, no matter which version of the code we use.

------
## Further Analysis: Credential and Institutional Level



In [7]:
print(collegeScorecard["CREDLEV"].value_counts())
print(pseo["degree_level"].value_counts())
print(pseo["degree_level_label"].value_counts())

# Real institution + CIP match rate: check if (OPEID6, CIP_family) pairs exist in PSEO
pseo_keys = set(zip(pseo["OPEID6"], pseo["CIP_family"]))
collegeScorecard["has_institution_cip_match"] = collegeScorecard.apply(
    lambda row: (row["OPEID6"], row["CIP_family"]) in pseo_keys, axis=1
)

inst_match_pct = collegeScorecard["has_institution_cip_match"].mean() * 100
print(f"Real institution+CIP match rate: {inst_match_pct:.2f}%")

# Duplicate-key check: are there any (OPEID6, CIP_family) pairs with more than one PSEO row?
dupe_keys = pseo.groupby(["OPEID6", "CIP_family"]).size()
print("(OPEID6, CIP_family) pairs with more than 1 PSEO row:", (dupe_keys > 1).sum())

CREDLEV
1    26678
Name: count, dtype: int64
degree_level
1    10025
2     8322
4      479
Name: count, dtype: int64
degree_level_label
Certificate < 1 year     10025
Certificate 1-2 years     8322
Certificate 2-4 years      479
Name: count, dtype: int64
Real institution+CIP match rate: 17.37%
(OPEID6, CIP_family) pairs with more than 1 PSEO row: 4169


C:\Users\andre\AppData\Local\Temp\ipykernel_39000\2371501414.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  collegeScorecard["has_institution_cip_match"] = collegeScorecard.apply(


----------------
## PSEO + CS Merge: Summary

**Checked:** CIP code format, CIP code coverage, credential level alignment, and institution-level match rate.

**Findings:**
- CIP codes match well: 99.5% of our rows have a usable program-level match.
- Credential levels already line up (both datasets are certificate-only).
- But matching on the **actual school + program** drops coverage to **17.4%**: PSEO simply doesn't report data for most of our schools, and some of the pairs it does list are suppressed for privacy (too few graduates).

**Decision:** The Pell question needs **program-specific** metrics, so the school + program earnings are kept on their own and left as NA when PSEO doesn't provide them. National (cross-institution) averages go in **separate benchmark columns** for context only; they are never substituted into the program-specific columns.


--------------
## Start Merge

### Adding PSEO Earnings Data

Algorithm:
1. PSEO has multiple rows per school/program (one for each certificate length), so we first combine those into a single average at three levels: school + program, program only (national), and broader category (national).
2. We merge all three onto our data as **separate columns**.
3. The school + program columns (`y1_earnings_program`, `y5_earnings_program`) are **never filled in** from the national numbers. Where they're missing, a status column says why:
   - `reported`: PSEO has earnings for this exact school + program
   - `suppressed (not enough data)`: PSEO lists this school + program but withheld the earnings (too few graduates)
   - `not provided by PSEO`: this school + program doesn't appear in PSEO at all
4. The national benchmarks sit in their own columns, plus one combined benchmark column (program average, falling back to the broader category) with a label showing which level it came from.

In [8]:
pseo_inst_level = pseo.groupby(["OPEID6", "CIP_family"], as_index=False).agg(
    y1_earnings_program=("y1_p50_earnings", "mean"),
    y5_earnings_program=("y5_p50_earnings", "mean"),
)
pseo_inst_level["in_pseo_program"] = True  # marks that the school + program appears in PSEO at all

pseo_cip_level = pseo.groupby("CIP_family", as_index=False).agg(
    y1_earnings_natl_cip_family=("y1_p50_earnings", "mean"),
    y5_earnings_natl_cip_family=("y5_p50_earnings", "mean"),
)

pseo_series_level = pseo.groupby("CIP_series", as_index=False).agg(
    y1_earnings_natl_cip_series=("y1_p50_earnings", "mean"),
    y5_earnings_natl_cip_series=("y5_p50_earnings", "mean"),
)

**Merge into College Scorecard**

Attach all three PSEO levels as separate columns: school + program, national program average, and national broader-category average.

In [9]:
collegeScorecard = collegeScorecard.merge(pseo_inst_level, on=["OPEID6", "CIP_family"], how="left")
collegeScorecard = collegeScorecard.merge(pseo_cip_level, on="CIP_family", how="left")
collegeScorecard = collegeScorecard.merge(pseo_series_level, on="CIP_series", how="left")

collegeScorecard["in_pseo_program"] = collegeScorecard["in_pseo_program"].fillna(False).astype(bool)

**Label the program-specific data and build the national benchmark**

- Program-specific earnings stay as-is (NA when missing), with a status column explaining each NA.
- The combined national benchmark uses the national program average, falling back to the broader category, and records which level it used.

In [10]:
def program_status(earn_col):
    return np.select(
        [
            collegeScorecard[earn_col].notna(),
            collegeScorecard["in_pseo_program"],
        ],
        ["reported", "suppressed (not enough data)"],
        default="not provided by PSEO",
    )

# Program-specific (school + program): no fill-in, NA stays NA
collegeScorecard["y1_earnings_program_status"] = program_status("y1_earnings_program")
collegeScorecard["y5_earnings_program_status"] = program_status("y5_earnings_program")

# National benchmark: separate column, program average first, then broader category
for yr in ["y1", "y5"]:
    fam = f"{yr}_earnings_natl_cip_family"
    ser = f"{yr}_earnings_natl_cip_series"
    collegeScorecard[f"{yr}_earnings_natl_benchmark"] = collegeScorecard[fam].fillna(collegeScorecard[ser])
    collegeScorecard[f"{yr}_earnings_natl_benchmark_level"] = np.select(
        [collegeScorecard[fam].notna(), collegeScorecard[ser].notna()],
        ["cip_family", "cip_series"],
        default="none",
    )

collegeScorecard = collegeScorecard.drop(columns=["in_pseo_program"])

C:\Users\andre\AppData\Local\Temp\ipykernel_39000\2848909387.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  collegeScorecard["y1_earnings_program_status"] = program_status("y1_earnings_program")
C:\Users\andre\AppData\Local\Temp\ipykernel_39000\2848909387.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  collegeScorecard["y5_earnings_program_status"] = program_status("y5_earnings_program")
C:\Users\andre\AppData\Local\Temp\ipykernel_39000\2848909387.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usu

Check the results: program-specific status counts and national benchmark coverage.

In [11]:
for yr in ["y1", "y5"]:
    print(f"{yr.upper()} program-specific (school + program) status:")
    print(collegeScorecard[f"{yr}_earnings_program_status"].value_counts(), "\n")
    print(f"{yr.upper()} national benchmark level:")
    print(collegeScorecard[f"{yr}_earnings_natl_benchmark_level"].value_counts(), "\n")

earn_cols = [
    "y1_earnings_program", "y5_earnings_program",
    "y1_earnings_natl_cip_family", "y5_earnings_natl_cip_family",
    "y1_earnings_natl_cip_series", "y5_earnings_natl_cip_series",
    "y1_earnings_natl_benchmark", "y5_earnings_natl_benchmark",
]
print("Non-null counts:")
print(collegeScorecard[earn_cols].notna().sum())

Y1 program-specific (school + program) status:
y1_earnings_program_status
not provided by PSEO            22044
reported                         2798
suppressed (not enough data)     1836
Name: count, dtype: int64 

Y1 national benchmark level:
y1_earnings_natl_benchmark_level
cip_family    25834
cip_series      716
none            128
Name: count, dtype: int64 

Y5 program-specific (school + program) status:
y5_earnings_program_status
not provided by PSEO            22044
suppressed (not enough data)     2391
reported                         2243
Name: count, dtype: int64 

Y5 national benchmark level:
y5_earnings_natl_benchmark_level
cip_family    25504
cip_series     1039
none            135
Name: count, dtype: int64 

Non-null counts:
y1_earnings_program             2798
y5_earnings_program             2243
y1_earnings_natl_cip_family    25834
y5_earnings_natl_cip_family    25504
y1_earnings_natl_cip_series    26550
y5_earnings_natl_cip_series    26543
y1_earnings_natl_benchmark   

## Merge Results - Summary

**New earnings columns**

| Column | What it is | Missing values |
|---|---|---|
| `y1_earnings_program`, `y5_earnings_program` | Median earnings for this exact school + program (**use these for the Pell analysis**) | NA, explained by the status column |
| `y1_earnings_program_status`, `y5_earnings_program_status` | `reported` / `suppressed (not enough data)` / `not provided by PSEO` | none |
| `y1_earnings_natl_cip_family`, `y5_earnings_natl_cip_family` | National average for the 4-digit program (e.g. `16.09`) | NA if PSEO has no data for that program |
| `y1_earnings_natl_cip_series`, `y5_earnings_natl_cip_series` | National average for the 2-digit field (e.g. `16`) | NA if PSEO has no data for that field |
| `y1_earnings_natl_benchmark`, `y5_earnings_natl_benchmark` | Program national average, falling back to the field average | NA only when neither exists |
| `y1_earnings_natl_benchmark_level`, `y5_earnings_natl_benchmark_level` | `cip_family` / `cip_series` / `none` | none |

**Data notes**
- **Program-specific coverage is low (~10% of rows for year 1)** because PSEO only covers some states and schools and suppresses small programs. Those rows are left as NA on purpose; don't fill them with the national numbers.
- **National averages are context, not program results.** Use them to compare a program against its field, not as a stand-in for that program's outcomes.
- **"National" here means across all PSEO-reporting schools**, which is not every school in the U.S.
- **How the averages were calculated**: national numbers are unweighted averages of each program's median earnings, so a program with 5 graduates counts the same as one with 500.

--------------
## Save Output File - MERGED

In [12]:
output_path = "../../../data/merges/CS_PSEO_merged.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

collegeScorecard.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")
print("Final shape:", collegeScorecard.shape)

Saved cleaned dataset to: ../../../data/merges/CS_PSEO_merged.csv
Final shape: (26678, 122)
